In [1]:
#PART A

import pandas as pd
import numpy as np
import time

# 1. Load the dataset
df = pd.read_csv('garments_worker_productivity.csv')

# 2. Clean 'department' whitespace and typo ('sweing' -> 'sewing')
df['department'] = df['department'].str.strip().replace({'sweing': 'sewing'})

# 3. Handle missing values in 'wip' (Work In Progress)
# Missing WIP occurs mainly in finishing lines where WIP is not tracked -> fill with 0
df['wip'] = df['wip'].fillna(0)

# 4. Create Classification Target: MeetsTarget (1 if actual >= targeted, else 0)
df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

# 5. Drop 'date' (we already have 'day' and 'quarter')
if 'date' in df.columns:
    df = df.drop(columns=['date'])

print("Cleaned Data Shape:", df.shape)
print("\nTarget Class Distribution (MeetsTarget):")
print(df['MeetsTarget'].value_counts())
df.head(3)

Cleaned Data Shape: (1197, 15)

Target Class Distribution (MeetsTarget):
MeetsTarget
1    875
0    322
Name: count, dtype: int64


,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity,MeetsTarget
0,Quarter1,sewing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,0.0,960,0,0.0,0,0,8.0,0.886500,1
2,Quarter1,sewing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Separate Features (X) and Targets (y_reg for regression, y_clf for classification)
# Notice: 'actual_productivity' and 'MeetsTarget' are both dropped from X!
X = df.drop(columns=['actual_productivity', 'MeetsTarget'])
y_reg = df['actual_productivity']
y_clf = df['MeetsTarget']

# Identify categorical and numerical columns
categorical_cols = ['quarter', 'department', 'day', 'team']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Create preprocessor: OneHot for categories, StandardScaler for numbers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)

# Fixed Train-Test Split (80% Train, 20% Test)
# We split X, y_reg, and y_clf simultaneously using the same random_state
X_train_raw, X_test_raw, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# Fit preprocessor ONLY on training data, then transform both train and test (prevents data snooping!)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Training feature shape: {X_train.shape}")
print(f"Testing feature shape: {X_test.shape}")

Training feature shape: (957, 30)
Testing feature shape: (240, 30)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Initialize Model
sklearn_lr = LinearRegression()

# 2. Train and measure training time
start_train = time.perf_counter()
sklearn_lr.fit(X_train, y_train_reg)
sklearn_lr_train_time = time.perf_counter() - start_train

# 3. Predict and measure prediction time
start_pred = time.perf_counter()
y_pred_reg_sklearn = sklearn_lr.predict(X_test)
sklearn_lr_pred_time = time.perf_counter() - start_pred

# 4. Evaluate Metrics
mae_sklearn = mean_absolute_error(y_test_reg, y_pred_reg_sklearn)
rmse_sklearn = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_sklearn))
r2_sklearn = r2_score(y_test_reg, y_pred_reg_sklearn)

print("--- Scikit-Learn Linear Regression Results ---")
print(f"Training Time   : {sklearn_lr_train_time:.6f} seconds")
print(f"Prediction Time : {sklearn_lr_pred_time:.6f} seconds")
print(f"MAE             : {mae_sklearn:.4f}")
print(f"RMSE            : {rmse_sklearn:.4f}")
print(f"R2 Score        : {r2_sklearn:.4f}")

--- Scikit-Learn Linear Regression Results ---
Training Time   : 0.026809 seconds
Prediction Time : 0.001491 seconds
MAE             : 0.1106
RMSE            : 0.1495
R2 Score        : 0.1584


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Initialize Model
sklearn_logreg = LogisticRegression(max_iter=1000, random_state=42)

# 2. Train and measure training time
start_train = time.perf_counter()
sklearn_logreg.fit(X_train, y_train_clf)
sklearn_logreg_train_time = time.perf_counter() - start_train

# 3. Predict and measure prediction time
start_pred = time.perf_counter()
y_pred_clf_sklearn = sklearn_logreg.predict(X_test)
sklearn_logreg_pred_time = time.perf_counter() - start_pred

# 4. Evaluate Metrics
acc_sklearn = accuracy_score(y_test_clf, y_pred_clf_sklearn)
prec_sklearn = precision_score(y_test_clf, y_pred_clf_sklearn)
rec_sklearn = recall_score(y_test_clf, y_pred_clf_sklearn)
f1_sklearn = f1_score(y_test_clf, y_pred_clf_sklearn)

print("--- Scikit-Learn Logistic Regression Results ---")
print(f"Training Time   : {sklearn_logreg_train_time:.6f} seconds")
print(f"Prediction Time : {sklearn_logreg_pred_time:.6f} seconds")
print(f"Accuracy        : {acc_sklearn:.4f}")
print(f"Precision       : {prec_sklearn:.4f}")
print(f"Recall          : {rec_sklearn:.4f}")
print(f"F1 Score        : {f1_sklearn:.4f}")

--- Scikit-Learn Logistic Regression Results ---
Training Time   : 0.042235 seconds
Prediction Time : 0.000643 seconds
Accuracy        : 0.7583
Precision       : 0.7874
Recall          : 0.9209
F1 Score        : 0.8490


In [ ]:
#PART B

